# Reconstruct 5 IPv4 Graphs (Compressed from 15) with RTT

This Google Colab notebook compresses the original 15 small IPv4 graphs (~400 nodes each) 3-to-1 into **5 larger graphs (1,096 - 1,372 nodes each)**.

### Architectural Standards:
- **Zero Target Leakage**: True hidden-path RTT is stored in `path_ground_truth.csv` and never in model inputs.
- **Tomography Mathematical Consistency**: In every graph, row sums of the incidence matrix $A$ strictly equal the path hop count ($A_{m, \cdot} \mathbf{1} = \text{hops}_m$).
- **Train-Only Normalization**: Node statistics, edge statistics, and log-RTT parameters are fitted strictly on `train` graphs (`graph_01`, `graph_02`, `graph_03`).
- **Native DeepNT+ Compatibility**: Loads directly into `deepnt_plus.data.load_dataset()`.


In [ ]:
# 0. Google Colab Drive Mount & Path Auto-Detection
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path('/content/drive/My Drive/Thesis')
else:
    THESIS_DIR = Path(r'C:\Users\KINGDOM\Desktop\Thesis')

PROJECT_ROOT = THESIS_DIR / 'DeepNT_Output'
OUT_DIR = PROJECT_ROOT / 'reconstructed_dataset' / 'ipv4'
PROCESSED_ACTIVE = PROJECT_ROOT / 'data' / 'processed'

# Add colab package to path
sys.path.insert(0, str(THESIS_DIR / 'colab'))

for d in [OUT_DIR / 'graphs', OUT_DIR / 'tomography', OUT_DIR / 'tgn']:
    d.mkdir(parents=True, exist_ok=True)

print('Environment initialized successfully.')
print(f'Target Output Directory: {OUT_DIR}')


In [ ]:
# 1. Load the 15 original IPv4 graphs
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
import json
from collections import Counter

backup_dir = PROJECT_ROOT / 'data' / 'processed_15graphs_backup'
active_dir = PROJECT_ROOT / 'data' / 'processed'
if backup_dir.exists() and (backup_dir / 'paths_all.csv').exists():
    SRC_PROCESSED = backup_dir
elif active_dir.exists() and (active_dir / 'paths_all.csv').exists():
    SRC_PROCESSED = active_dir
else:
    raise FileNotFoundError(f'Cannot find source 15 graphs in {backup_dir} or {active_dir}')

paths_all_src = pd.read_csv(SRC_PROCESSED / 'paths_all.csv')
gt_src = pd.read_csv(SRC_PROCESSED / 'path_ground_truth.csv')
print(f'Loaded {len(paths_all_src):,} paths across {paths_all_src["graph_id"].nunique()} original graphs.')

# Compression mapping: 3 original graphs merged into 1 updated graph
COMPRESSION_MAP = {
    'graph_01': 'graph_01', 'graph_02': 'graph_01', 'graph_03': 'graph_01',
    'graph_04': 'graph_02', 'graph_05': 'graph_02', 'graph_06': 'graph_02',
    'graph_07': 'graph_03', 'graph_08': 'graph_03', 'graph_09': 'graph_03',
    'graph_10': 'graph_04', 'graph_11': 'graph_04', 'graph_12': 'graph_04',
    'graph_13': 'graph_05', 'graph_14': 'graph_05', 'graph_15': 'graph_05',
}
SPLIT_MAP_V4 = {
    'graph_01': 'train', 'graph_02': 'train', 'graph_03': 'train',
    'graph_04': 'val', 'graph_05': 'test'
}
GRAPH_IDS_V4 = sorted(set(COMPRESSION_MAP.values()))

merged_v4 = paths_all_src.merge(gt_src[['path_id', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'quality_flag']], on='path_id')
merged_v4['orig_graph_id'] = merged_v4['graph_id']
merged_v4['graph_id'] = merged_v4['orig_graph_id'].map(COMPRESSION_MAP)
merged_v4['split'] = merged_v4['graph_id'].map(SPLIT_MAP_V4)
print('Monitors per compressed graph:')
print(merged_v4.groupby(['graph_id', 'monitor_id']).size().unstack().fillna(0).astype(int))


In [ ]:
# 2. Build 5 merged NetworkX topologies & Assign 45% Observation Masks
SEED = 2026
OBSERVATION_RATE = 0.45
RTT_MIN_MS, RTT_MAX_MS = 0.05, 1000.0

graphs_v4_nx = {}
path_rows_v4 = []

for gid in GRAPH_IDS_V4:
    sub_df = merged_v4[merged_v4['graph_id'] == gid].reset_index(drop=True)
    if len(sub_df) == 0:
        raise ValueError(f'No paths found for {gid}. Ensure SRC_PROCESSED has all 15 original graphs.')
    G = nx.Graph()
    clean_items = []
    for _, r in sub_df.iterrows():
        nodes = r['path_nodes'].split('|')
        edges = [tuple(e.split('-')) for e in r['path_edges'].split('|')]
        for a, b in edges:
            if not G.has_edge(a, b):
                G.add_edge(a, b, edge_id=f'{a}-{b}', path_count=0)
            G[a][b]['path_count'] += 1
        clean_items.append({
            'nodes': nodes, 'edges': edges, 'rtt_ms': float(r['rtt_ms']),
            'rtt_raw': float(r['rtt_raw']), 'rtt_raw_unit': r['rtt_raw_unit'],
            'monitor_id': r['monitor_id'], 'measurement_time': r['measurement_time'],
            'measurement_method': r['measurement_method'], 'raw_id': r['raw_measurement_id'],
            'rtt_source': r['rtt_source'], 'structural_flag': r['quality_flag_x'],
            'quality_flag': r['quality_flag_y'],
        })
    graphs_v4_nx[gid] = G
    n_obs = max(1, int(round(OBSERVATION_RATE * len(clean_items))))
    observed = np.zeros(len(clean_items), dtype=int)
    g_num = int(gid.split('_')[1])
    observed[np.random.default_rng(SEED + g_num).choice(len(clean_items), n_obs, replace=False)] = 1
    
    for k, it in enumerate(clean_items):
        rtt_val = it['rtt_ms']
        outlier = [] if RTT_MIN_MS <= rtt_val <= RTT_MAX_MS else (['rtt_outlier_high'] if rtt_val > RTT_MAX_MS else ['rtt_outlier_low'])
        path_rows_v4.append({
            'graph_id': gid, 'path_id': f'{gid}_p{k:05d}', 'split': SPLIT_MAP_V4[gid],
            'source_node': it['nodes'][0], 'destination_node': it['nodes'][-1],
            'path_nodes': '|'.join(it['nodes']),
            'path_edges': '|'.join(f'{a}-{b}' for a, b in it['edges']),
            'path_length': len(it['edges']),
            'measurement_time': it['measurement_time'], 'monitor_id': it['monitor_id'],
            'measurement_method': it['measurement_method'], 'observed': int(observed[k]),
            'raw_measurement_id': it['raw_id'], 'rtt_source': it['rtt_source'],
            'structural_flag': it['structural_flag'],
            'quality_flag': ';'.join([f for f in str(it['quality_flag']).split(';') if f != 'ok'] + outlier) or 'ok',
            'rtt_ms': rtt_val, 'rtt_raw': it['rtt_raw'], 'rtt_raw_unit': it['rtt_raw_unit'],
        })

full_v4 = pd.DataFrame(path_rows_v4)
print('5 Compressed IPv4 Graph Sizes:')
for gid in GRAPH_IDS_V4:
    print(f'  {gid}: {graphs_v4_nx[gid].number_of_nodes():,} nodes, {graphs_v4_nx[gid].number_of_edges():,} edges')


In [ ]:
# 3. Compute 8 Structural Node Features & Edge Features
print('Computing node topological features (degree, pagerank, centrality, k-core)...')
node_rows_v4 = []
for gid, G in graphs_v4_nx.items():
    n = G.number_of_nodes()
    deg, clus, pr, core = dict(G.degree()), nx.clustering(G), nx.pagerank(G), nx.core_number(G)
    clo = nx.closeness_centrality(G) if n <= 5000 else {}
    bet = nx.betweenness_centrality(G, k=min(n, 300), seed=SEED)
    for node in G:
        node_rows_v4.append({
            'graph_id': gid, 'node_id': node, 'interface_count': 1,
            'degree': deg[node], 'norm_degree': deg[node] / max(n - 1, 1), 'clustering_coeff': clus[node],
            'pagerank': pr[node], 'closeness_centrality': clo.get(node, np.nan),
            'betweenness_centrality': bet[node], 'k_core': core[node],
            'asn': pd.NA, 'country': pd.NA, 'latitude': np.nan, 'longitude': np.nan
        })
node_features_v4 = pd.DataFrame(node_rows_v4)

obs_count_v4 = Counter()
for r in full_v4[full_v4['observed'] == 1].itertuples():
    for e in r.path_edges.split('|'):
        obs_count_v4[(r.graph_id, e)] += 1

edge_rows_v4 = []
for gid, G in graphs_v4_nx.items():
    for _, _, d in sorted(G.edges(data=True), key=lambda x: x[2]['edge_id']):
        parts = d['edge_id'].split('-')
        edge_rows_v4.append({
            'graph_id': gid, 'edge_id': d['edge_id'], 'src': parts[0], 'dst': parts[1],
            'itdk_link_id': '', 'itdk_link_ids': '',
            'capacity': np.nan, 'utilization': np.nan, 'base_delay': np.nan, 'loss_rate': np.nan,
            'path_count_all': d['path_count'], 'observed_measurement_count': obs_count_v4[(gid, d['edge_id'])],
        })
edge_features_v4 = pd.DataFrame(edge_rows_v4)
print(f'Computed features for {len(node_features_v4):,} nodes and {len(edge_features_v4):,} edges.')


In [ ]:
# 4. Build Tomography A matrices, TGN events, Train Normalization & Save Files
STRUCT = ['graph_id', 'path_id', 'split', 'source_node', 'destination_node',
          'path_nodes', 'path_edges', 'path_length', 'measurement_time',
          'monitor_id', 'measurement_method', 'observed', 'raw_measurement_id']

paths_all_v4 = full_v4[STRUCT + ['rtt_source']].assign(quality_flag=full_v4['structural_flag'])
gt_v4 = full_v4[['path_id', 'graph_id', 'split', 'observed', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'rtt_source', 'quality_flag']].assign(
    packet_loss=np.nan, loss_flag='unavailable_no_probe_counts')
paths_obs_v4 = paths_all_v4[paths_all_v4['observed'] == 1].merge(
    gt_v4[['path_id', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'packet_loss', 'loss_flag']], on='path_id')
paths_unobs_v4 = paths_all_v4[paths_all_v4['observed'] == 0]

A_blocks_v4, row_idx_v4, col_idx_v4 = [], [], []
for gid in GRAPH_IDS_V4:
    prow = paths_all_v4[paths_all_v4['graph_id'] == gid].reset_index(drop=True)
    ecol = edge_features_v4[edge_features_v4['graph_id'] == gid].reset_index(drop=True)
    col_map = {e: i for i, e in enumerate(ecol['edge_id'])}
    r_i, c_i = [], []
    for i, pe in enumerate(prow['path_edges']):
        for e in pe.split('|'):
            r_i.append(i)
            c_i.append(col_map[e])
    A = sp.csr_matrix((np.ones(len(r_i), dtype=np.int8), (r_i, c_i)), shape=(len(prow), len(ecol)))
    assert (np.asarray(A.sum(axis=1)).ravel() == prow['path_length'].to_numpy()).all(), f'A check failed on {gid}'
    sp.save_npz(OUT_DIR / 'tomography' / f'A_{gid}.npz', A)
    A_blocks_v4.append(A)
    row_idx_v4 += [(gid, p, i) for i, p in enumerate(prow['path_id'])]
    col_idx_v4 += [(gid, e, i) for i, e in enumerate(ecol['edge_id'])]

sp.save_npz(OUT_DIR / 'tomography' / 'A_matrix.npz', sp.block_diag(A_blocks_v4, format='csr'))
pd.DataFrame(row_idx_v4, columns=['graph_id', 'path_id', 'row']).to_csv(OUT_DIR / 'tomography' / 'A_rows.csv', index=False)
pd.DataFrame(col_idx_v4, columns=['graph_id', 'edge_id', 'col']).to_csv(OUT_DIR / 'tomography' / 'A_cols.csv', index=False)

# TGN Events
ev_v4 = paths_obs_v4[paths_obs_v4['measurement_time'] != ''].sort_values('measurement_time')
events_v4 = pd.DataFrame({
    'event_id': [f'ev_{i:06d}' for i in range(len(ev_v4))],
    'timestamp': ev_v4['measurement_time'].to_numpy(),
    'source_node': ev_v4['source_node'].to_numpy(),
    'destination_node': ev_v4['destination_node'].to_numpy(),
    'event_type': 'path_rtt_measurement',
    'value_rtt_ms': ev_v4['rtt_ms'].to_numpy(),
    'provenance': ev_v4['rtt_source'].to_numpy(),
    'quality_flag': ev_v4['quality_flag'].to_numpy(),
    'graph_id': ev_v4['graph_id'].to_numpy(),
    'path_id': ev_v4['path_id'].to_numpy()
})
events_v4.to_csv(OUT_DIR / 'tgn' / 'events.csv', index=False)

# Train-only Normalization (Graphs 01, 02, 03)
feat_cols = ['interface_count', 'degree', 'norm_degree', 'clustering_coeff', 'pagerank',
             'closeness_centrality', 'betweenness_centrality', 'k_core']
train_nodes_v4 = node_features_v4[node_features_v4['graph_id'].map(SPLIT_MAP_V4) == 'train']
train_y_v4 = np.log1p(paths_obs_v4.loc[
    (paths_obs_v4['split'] == 'train') & (~paths_obs_v4['quality_flag'].astype(str).str.contains('rtt_outlier')), 'rtt_ms'
])
norm_v4 = {
    'fit_on': 'train graphs only (graphs 1, 2, 3)',
    'node_features': {c: {'mean': float(train_nodes_v4[c].mean()), 'std': float(train_nodes_v4[c].std() or 1.0)} for c in feat_cols},
    'log1p_rtt_ms': {'mean': float(train_y_v4.mean()), 'std': float(train_y_v4.std())}
}
(OUT_DIR / 'normalization.json').write_text(json.dumps(norm_v4, indent=2))

# Save Tables
for gid, G in graphs_v4_nx.items():
    sub_edges = edge_features_v4[edge_features_v4['graph_id'] == gid][['src', 'dst', 'edge_id', 'itdk_link_id']]
    sub_edges.rename(columns={'src': 'source', 'dst': 'target'}).to_csv(OUT_DIR / 'graphs' / f'{gid}.csv', index=False)

node_features_v4.to_csv(OUT_DIR / 'node_features.csv', index=False)
edge_features_v4.to_csv(OUT_DIR / 'edge_features.csv', index=False)
paths_all_v4.to_csv(OUT_DIR / 'paths_all.csv', index=False)
paths_obs_v4.to_csv(OUT_DIR / 'paths_observed.csv', index=False)
paths_unobs_v4.to_csv(OUT_DIR / 'paths_unobserved.csv', index=False)
gt_v4.to_csv(OUT_DIR / 'path_ground_truth.csv', index=False)

# Also update data/processed for immediate training use
import shutil
for f in ['node_features.csv', 'edge_features.csv', 'paths_all.csv', 'paths_observed.csv', 'paths_unobserved.csv', 'path_ground_truth.csv', 'normalization.json']:
    shutil.copy2(OUT_DIR / f, PROCESSED_ACTIVE / f)
for sub in ['graphs', 'tomography', 'tgn']:
    shutil.copytree(OUT_DIR / sub, PROCESSED_ACTIVE / sub, dirs_exist_ok=True)

print(f'Successfully saved IPv4 dataset to {OUT_DIR} and deployed to {PROCESSED_ACTIVE}')


In [ ]:
# 5. Verify Dataset with Native DeepNT+ Loader
from deepnt_plus.data import load_dataset, describe
graphs, norm = load_dataset(OUT_DIR)
df_desc = describe(graphs)
print(df_desc.to_string(index=False))
